Step 2-keep only useful columns and drop rows with empty or irrelevant SYMPTOM_TEXT.

In [1]:
import pandas as pd
import os

# Paths relative to notebooks/ folder
data_path_processed = "../data/processed"

INPUT  = os.path.join(data_path_processed, "combined_covid_vaers.csv")
OUTPUT = os.path.join(data_path_processed, "combined_covid_vaers_relevant.csv")

# Keep only these useful columns (added AGE_YRS)
usecols = [
    "VAERS_ID", "SYMPTOM_TEXT",
    "SYMPTOM1", "SYMPTOM2", "SYMPTOM3", "SYMPTOM4", "SYMPTOM5",
    "DIED", "L_THREAT", "ER_VISIT", "HOSPITAL", "DISABLE", "RECOVD",
    "VAX_NAME", "VAX_DATE", "ONSET_DATE", "NUMDAYS", "YEAR",
    "AGE_YRS",
]

# Patterns that mark irrelevant symptom texts
irrelevant_patterns = [
    "no adverse event", "none reported", "n/a",
    "not applicable", "no symptoms", "none",
]

def is_irrelevant(text: str) -> bool:
    if pd.isna(text):
        return True
    t = str(text).strip().lower()
    return any(pat in t for pat in irrelevant_patterns)

# Process in chunks
first = True
for chunk in pd.read_csv(
    INPUT,
    usecols=usecols,
    chunksize=150_000,
    dtype={"VAERS_ID": "string"},
    low_memory=False
):
    # Drop rows with empty/irrelevant SYMPTOM_TEXT
    chunk = chunk.dropna(subset=["SYMPTOM_TEXT"])
    chunk = chunk[~chunk["SYMPTOM_TEXT"].apply(is_irrelevant)]

    # Drop duplicates within the chunk
    chunk = chunk.drop_duplicates(subset=["VAERS_ID", "SYMPTOM_TEXT"])

    # Append to output
    chunk.to_csv(OUTPUT, index=False, mode="w" if first else "a", header=first)
    first = False

print(f"✅ Saved filtered relevant COVID-19 VAERS data → {OUTPUT}")


✅ Saved filtered relevant COVID-19 VAERS data → ../data/processed\combined_covid_vaers_relevant.csv


In [ ]:
import pandas as pd
print(pd.__version__)


2.3.2


In [12]:
import sys, os, glob, pandas as pd
print("Loaded from:", pd.__file__)
print("First 5 sys.path entries:", sys.path[:5])
print("Local things named 'pandas*' in CWD:", [p for p in glob.glob("pandas*") if os.path.exists(p)])


AttributeError: partially initialized module 'pandas' from 'c:\Users\Mochitha vijayan\ds-rpc-02\venv\Lib\site-packages\pandas\__init__.py' has no attribute '_pandas_datetime_CAPI' (most likely due to a circular import)